# ACT 학습 — my_task (SO-101) · Colab

**실행 전 2가지:**

1. **런타임 → 런타임 유형 변경 → T4 GPU** (안 하면 CPU로 돌아간다)
2. **왼쪽 열쇠 아이콘(보안 비밀) → `HF_TOKEN` 추가** — Hugging Face **write** 토큰
   (https://huggingface.co/settings/tokens · "노트북 액세스" 토글을 켤 것)

**Colab의 핵심 문제는 세션이 끊긴다는 것이다.** `/content`는 런타임이 죽으면 사라지므로,
이 노트북은 체크포인트를 **Google Drive**에 저장한다. 끊겨도 셀 5로 이어서 학습하면 된다.

## 1. 환경 확인

In [ ]:
import sys, subprocess
print("python:", sys.version.split()[0])
try:
    import torch
    print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print(p.name, round(p.total_memory/2**30, 1), "GB | sm", f"{p.major}.{p.minor}",
              "| bf16:", torch.cuda.is_bf16_supported())
except ImportError:
    print("torch 미설치")
subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total",
                "--format=csv,noheader"])

> **`cuda: False`가 나오면** 런타임 → 런타임 유형 변경 → T4 GPU 로 바꾸고 다시 실행하세요.
>
> **T4는 Turing(sm 7.5)이라 bf16을 지원하지 않습니다.** 위 출력의 `bf16` 값을 보고
> 셀 4에서 `MIXED_PRECISION`을 자동으로 맞춥니다.

## 2. Google Drive 연결 — 체크포인트를 여기에 저장한다

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

OUT = Path("/content/drive/MyDrive/lerobot/act_run")
OUT.parent.mkdir(parents=True, exist_ok=True)
print("output_dir =", OUT)
print("이미 있는 체크포인트:", sorted(p.name for p in (OUT/"checkpoints").glob("0*")) or "없음")

## 3. lerobot 설치

lerobot은 **Python 3.12 이상**이 필요하다. Colab 이미지가 그보다 낮으면
`uv`로 3.12 가상환경을 따로 만든다(torch를 새로 받으므로 5~10분).
어느 쪽이든 `TRAIN` 변수에 실행 경로가 담긴다.

In [ ]:
import sys
from IPython import get_ipython
from pathlib import Path

PKG = "lerobot[training] @ git+https://github.com/huggingface/lerobot.git"

def run(cmd):
    print("$", " ".join(map(str, cmd)), flush=True)
    get_ipython().system(" ".join(map(str, cmd)))

if sys.version_info >= (3, 12):
    # Colab의 기존 torch(드라이버에 맞게 빌드된 것)를 그대로 쓴다.
    run([sys.executable, "-m", "pip", "install", "-q", PKG])
    TRAIN = "lerobot-train"
else:
    run([sys.executable, "-m", "pip", "install", "-q", "uv"])
    VENV = Path("/content/venv")
    run(["uv", "venv", "--python", "3.12", str(VENV)])
    run(["uv", "pip", "install", "--python", str(VENV/"bin"/"python"), PKG])
    TRAIN = str(VENV / "bin" / "lerobot-train")

print("\nTRAIN =", TRAIN)

## 4. Hugging Face 인증

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

from huggingface_hub import HfApi
print("logged in as:", HfApi().whoami()["name"])

## 5. 학습

`STEPS`를 조절해서 세션 안에 끝날 만큼만 돌린다. 무료 티어는 유휴 감지로 자주 끊기므로
**2만~3만 스텝씩 나눠 돌리고 셀 6으로 이어가는 쪽**이 현실적이다.

체크포인트가 Drive에 저장되므로 런타임이 죽어도 남는다. Drive 무료 용량은 15GB이고
체크포인트 1개가 591MB라, `SAVE_FREQ`를 너무 짧게 두면 금방 찬다.

In [ ]:
import torch
from IPython import get_ipython

DATASET = "<내-HF-아이디>/<데이터셋-이름>"
STEPS      = 30_000     # 세션 안에 끝날 만큼. 이어서 돌리면 되니 욕심내지 말 것
SAVE_FREQ  = 5_000
BATCH      = 8
LOG_FREQ   = 100        # 이 간격마다 loss/속도 한 줄이 찍힌다

# T4(Turing)는 bf16 미지원 -> fp16. L4/A100 등 Ampere 이상이면 bf16.
# including_emulation=False 가 중요하다. 기본값(True)이면 T4처럼 bf16 하드웨어가
# 없는 GPU에서도 True를 돌려주고, 느린 에뮬레이션 경로로 학습이 돈다.
MIXED_PRECISION = "bf16" if torch.cuda.is_bf16_supported(including_emulation=False) else "fp16"
print("mixed_precision =", MIXED_PRECISION)

cmd = " ".join([
    TRAIN,
    f"--dataset.repo_id={DATASET}",
    "--policy.type=act",
    "--policy.device=cuda",
    "--policy.push_to_hub=false",
    f"--accelerator.mixed_precision={MIXED_PRECISION}",
    "--dataset.return_uint8=true",
    f"--batch_size={BATCH}",
    f"--steps={STEPS}",
    f"--save_freq={SAVE_FREQ}",
    f"--log_freq={LOG_FREQ}",
    "--num_workers=2",
    f"--output_dir={OUT}",
    "--job_name=act_train",
])
print(cmd, "\n", flush=True)

# subprocess.run 은 OS 레벨 fd 로 출력을 내보내서 Colab 에서는 셀이 아니라
# 컨테이너 로그로 간다(= 화면에 아무것도 안 보인다).
# get_ipython().system() 은 IPython 이 파이프로 받아 셀에 실시간으로 흘려준다.
get_ipython().system(cmd)

## 6. 세션이 끊겼을 때 — 이어서 학습

**새 세션에서 셀 1~4를 실행한 뒤 이 셀을 실행한다** (셀 5는 건너뛴다).
Drive에 있는 `checkpoints/last`에서 이어받는다.

더 돌리고 싶으면 `--steps`를 늘려서 같이 넘기면 된다 (예: 30000 -> 60000).

In [ ]:
from IPython import get_ipython

CONFIG = OUT / "checkpoints" / "last" / "pretrained_model" / "train_config.json"
assert CONFIG.exists(), f"체크포인트가 없다: {CONFIG}"

get_ipython().system(
    f"{TRAIN} --config_path={CONFIG} --resume=true --steps=60000"
)  # --steps 는 이전보다 큰 값으로. 그대로 두면 즉시 끝난다

## 7. 학습된 모델 내려받기

로봇이 있는 컴퓨터에서 쓰려면 Hub에 올리는 게 가장 편하다 (198MB).

In [ ]:
from huggingface_hub import HfApi

REPO = "<내-HF-아이디>/<모델-이름>"
CKPT = OUT / "checkpoints" / "last" / "pretrained_model"

api = HfApi()
api.create_repo(REPO, repo_type="model", exist_ok=True)
api.upload_folder(folder_path=str(CKPT), repo_id=REPO, repo_type="model")
print("업로드 완료:", f"https://huggingface.co/{REPO}")

그 다음 로봇이 있는 컴퓨터에서:

```powershell
lerobot-rollout `
    --strategy.type=base `
    --policy.path=<내-HF-아이디>/<모델-이름> `
    --robot.type=so101_follower `
    --robot.port=COM6 `
    --robot.id=follower `
    --robot.cameras='{ top: {type: opencv, index_or_path: 0, width: 640, height: 480, fps: 30} }' `
    --task="Pick up things" `
    --duration=10
```

첫 실행은 `--duration=10`으로 짧게, 손을 비상정지 위치에 두고 할 것.

### Colab에서 알아둘 것

- **유휴 감지로 끊긴다.** 탭을 닫거나 오래 방치하면 런타임이 회수된다.
  한 번에 3만 스텝 이상 노리지 말고 나눠 돌릴 것.
- **`/content`는 사라지고 Drive는 남는다.** `output_dir`를 Drive로 둔 이유다.
  대신 Drive 쓰기는 로컬 디스크보다 느려서 체크포인트 저장에 20~60초씩 걸린다.
- **Drive 무료 용량 15GB.** 체크포인트 1개 591MB
  (모델 198MB + 옵티마이저 상태 394MB). 오래된 체크포인트는 지워가며 쓸 것.
- **T4는 bf16이 없다.** 셀 5가 자동으로 fp16으로 떨어뜨린다.